<a href="https://colab.research.google.com/github/ruben-antenen/bina/blob/main/wirtschaftlichkeitsrechnung.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modellbasierte Wirtschaftlichkeitsbewertung

## Berechnungen

In [90]:
import pandas as pd

# =============================
# 1) Parameter – Zentrale Konfiguration
# =============================
durchlaeufe_pro_jahr    = 4
schweine_pro_durchlauf  = 940
schweine_pro_bucht      = 12
buchten_total           = 80
abgangsrate             = 0.026
kosten_pro_schwein      = 640
praemie_pro_schwein     = 425
schaden_pro_abgang      = kosten_pro_schwein + praemie_pro_schwein
investition_pro_bucht   = 9500
lizenz_kosten_pro_jahr  = 5000

# Pilotdaten
buchten_pilot           = 4
investition_pilot_total = 38000
quartale_pilot          = ["Q4 2024","Q1 2025","Q2 2025","Q3 2025","Q4 2025"]
wirkungsgrad_pilot      = [0.0, 0.0, 0.2, 0.4, 0.6]

schweine_pilot          = buchten_pilot * schweine_pro_bucht
abgaenge_ohne_ki_pilot  = schweine_pilot * abgangsrate

# Rollout-Plan
n_ausbaujahre           = 3
n_folgejahre            = 12
jahre_rollout           = [f"{i+2026}" for i in range(n_ausbaujahre + n_folgejahre)]
delta_buchten           = buchten_total - buchten_pilot
buchten_jahresweise     = [
    int(round(buchten_pilot + (i+1)*delta_buchten/n_ausbaujahre))
    for i in range(n_ausbaujahre)
]
ausbauplan              = buchten_jahresweise + [buchten_total]*n_folgejahre
ki_wirkungsgrad         = wirkungsgrad_pilot[-1]

# =============================
# 2) Pilotphase (quartalsweise), Investition gleichmässig verteilt
# =============================
daten_pilot   = []
kumul_ersparnis = 0.0
invest_q = investition_pilot_total / len(quartale_pilot)

for i, q in enumerate(quartale_pilot):
    invest     = invest_q
    wg         = wirkungsgrad_pilot[i]
    verh_abg   = abgaenge_ohne_ki_pilot * wg
    ersp       = verh_abg * schaden_pro_abgang
    netto      = ersp - invest
    kumul_ersparnis += netto

    daten_pilot.append({
        "Phase":               "Pilot",
        "Periode":             q,
        "Investition":         invest,
        "Ersparnis":           ersp,
        "Kumulierte_Ersparnis": kumul_ersparnis
    })

# =============================
# 3) Rollout (jährlich)
# =============================
daten_rollout = []
buchten_last = buchten_pilot

for i, jahr in enumerate(jahre_rollout):
    bucht_aktuell = ausbauplan[i]
    neue_buchten  = bucht_aktuell - buchten_last if i < n_ausbaujahre else 0

    invest    = neue_buchten * investition_pro_bucht
    schweine  = bucht_aktuell * schweine_pro_bucht * durchlaeufe_pro_jahr
    verluste  = schweine * abgangsrate
    verh_abg  = verluste * ki_wirkungsgrad
    ersp      = verh_abg * schaden_pro_abgang
    netto     = ersp - invest - lizenz_kosten_pro_jahr
    kumul_ersparnis += netto

    daten_rollout.append({
        "Phase":               "Rollout",
        "Periode":             jahr,
        "Investition":         invest,
        "Ersparnis":           ersp,
        "Kumulierte_Ersparnis": kumul_ersparnis
    })
    buchten_last = bucht_aktuell

# =============================
# 4) Zusammenführen
# =============================
df_pilot   = pd.DataFrame(daten_pilot)
df_rollout = pd.DataFrame(daten_rollout)
df_gesamt  = pd.concat([df_pilot, df_rollout], ignore_index=True)

# Kontrolle
pd.set_option('display.float_format', lambda x: f"{x:,.2f}")
print(df_gesamt)

      Phase  Periode  Investition  Ersparnis  Kumulierte_Ersparnis
0     Pilot  Q4 2024     7,600.00       0.00             -7,600.00
1     Pilot  Q1 2025     7,600.00       0.00            -15,200.00
2     Pilot  Q2 2025     7,600.00     265.82            -22,534.18
3     Pilot  Q3 2025     7,600.00     531.65            -29,602.53
4     Pilot  Q4 2025     7,600.00     797.47            -36,405.06
5   Rollout     2026   237,500.00  23,126.69           -255,778.37
6   Rollout     2027   247,000.00  43,860.96           -463,917.41
7   Rollout     2028   237,500.00  63,797.76           -642,619.65
8   Rollout     2029         0.00  63,797.76           -583,821.89
9   Rollout     2030         0.00  63,797.76           -525,024.13
10  Rollout     2031         0.00  63,797.76           -466,226.37
11  Rollout     2032         0.00  63,797.76           -407,428.61
12  Rollout     2033         0.00  63,797.76           -348,630.85
13  Rollout     2034         0.00  63,797.76           -289,83

## Visualisierung

In [97]:
import pandas as pd
import altair as alt

# ———————————————————————————————————————————
# Konfigurierbare Formatparameter (minimalinvasiv)
# ———————————————————————————————————————————
TITLE_FONT_SIZE        = 18
AXIS_TITLE_FONT_SIZE   = 14
AXIS_LABEL_FONT_SIZE   = 14
LEGEND_TITLE_FONT_SIZE = 16
LEGEND_LABEL_FONT_SIZE = 14
BREAKEVEN_LABEL_FONT_SIZE = 14
EXPORT_SCALE_FACTOR    = 2   # für ca. 150 DPI beim Export

# ———————————————————————————————————————————
# 1) Ausgangsdaten: df_gesamt muss enthalten:
#    'Phase','Periode','Investition','Ersparnis','Kumulierte_Ersparnis'
# ———————————————————————————————————————————

# quartale_pilot = ["Pilot Q1","Pilot Q2","Pilot Q3","Pilot Q4","Pilot Q5"]
jahre_rollout  = [p for p in df_gesamt["Periode"] if p.startswith("20")]
perioden_order = quartale_pilot + jahre_rollout

df_gesamt["Periode"] = pd.Categorical(
    df_gesamt["Periode"],
    categories=perioden_order,
    ordered=True
)

df_long = df_gesamt.melt(
    id_vars=["Phase","Periode"],
    value_vars=["Investition","Ersparnis"],
    var_name="Typ", value_name="Betrag"
)
df_long["Plot"] = df_long.apply(
    lambda r: -r.Betrag if r.Typ=="Investition" else r.Betrag,
    axis=1
)

color_scale = alt.Scale(
    domain=["Investition","Ersparnis","Kumulierte_Ersparnis"],
    range=["#D62728","#276DC3","#33A02C"]
)

def phase_chart(phase_name):
    sub_long = df_long[df_long["Phase"]==phase_name]
    sub_df   = df_gesamt[df_gesamt["Phase"]==phase_name]

    bars = alt.Chart(sub_long).mark_bar(size=20).encode(
        x=alt.X("Periode:N", sort=perioden_order, title=None),
        y=alt.Y("Plot:Q", title="Cashflow (CHF)"),
        y2=alt.value(0),
        color=alt.Color("Typ:N", scale=color_scale, title="Typ")
    )

    line = alt.Chart(sub_df).mark_line(point=True).encode(
        x=alt.X("Periode:N", sort=perioden_order),
        y=alt.Y("Kumulierte_Ersparnis:Q", title="Kumulierte Ersparnis (CHF)"),
        color=alt.Color(value="#33A02C")
    )

    elems = [bars, line]
    if phase_name=="Rollout":
        idx_be = (sub_df["Kumulierte_Ersparnis"] >= 0).idxmax()
        be_per = sub_df.loc[idx_be, "Periode"]
        vline = alt.Chart(pd.DataFrame({"Periode":[be_per]})).mark_rule(
            color="orange", strokeDash=[4,2]
        ).encode(x=alt.X("Periode:N", sort=perioden_order))
        vtext = alt.Chart(pd.DataFrame({"Periode":[be_per]})).mark_text(
            text="Break-even", color="orange", dy=-10, fontWeight="bold", fontSize=BREAKEVEN_LABEL_FONT_SIZE
        ).encode(
            x=alt.X("Periode:N", sort=perioden_order),
            y=alt.value(0)
        )
        elems += [vline, vtext]

    return alt.layer(*elems).properties(
        width=900, height=200, title=phase_name
    )

# 2 Charts erzeugen
chart_pilot   = phase_chart("Pilot")
chart_rollout = phase_chart("Rollout")

# 3 Vertical Concat mit unabhängigen Y-Achsen und formatierte Achsen/Titel
final = alt.vconcat(
    chart_pilot,
    chart_rollout
).resolve_scale(y="independent") \
 .configure_title(fontSize=TITLE_FONT_SIZE, anchor='middle') \
 .configure_axis(titleFontSize=AXIS_TITLE_FONT_SIZE, labelFontSize=AXIS_LABEL_FONT_SIZE) \
 .configure_legend(titleFontSize=LEGEND_TITLE_FONT_SIZE, labelFontSize=LEGEND_LABEL_FONT_SIZE)

# Darstellung
final.display()

# Zum Export als PNG mit ~150 DPI:
# final.save('chart.png', scale=EXPORT_SCALE_FACTOR)

alt.VConcatChart(...)